# Лабораторна робота 7. Завдання 4. Власний набір даних Dogs vs Cats: підготовка, генератори, навчання CNN

Ноутбук 7.4. Виконав: Бабин Богдан Олегович, група 443-4

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**, потім «Выполнить все».

**План:** завантажуємо набір → власний скрипт `prepare_dataset.py` ділить його на train / val / test (70/15/15) →
генератори `ImageDataGenerator` → CNN з лекції → навчання з перевіркою і ранньою зупинкою → Adam проти RMSprop (lr=0.001) →
тестовий набір → збереження моделі → власні зображення, «складні» зображення та їх обрізані варіанти.

> Навчання двох моделей на 17 500 зображеннях займає приблизно **30–40 хвилин** на T4. Не закривайте вкладку.

In [ ]:
import os, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.utils import load_img, img_to_array
from google.colab import files
%matplotlib inline

tf.keras.utils.set_random_seed(42)

## 1. Завантаження набору Dogs vs Cats
Набір з https://www.kaggle.com/c/dogs-vs-cats (25 000 фото: 12 500 котів і 12 500 собак). Kaggle вимагає акаунт і API-ключ,
тому завантажуємо **той самий набір з офіційного дзеркала Microsoft** (без реєстрації, ~800 МБ, ~1 хв).

*Якщо хочете саме з Kaggle:* завантажте `kaggle.json` і виконайте `!kaggle competitions download -c dogs-vs-cats`,
`!unzip -q dogs-vs-cats.zip && unzip -q train.zip` — скрипт нижче підтримує і такий формат (`train/cat.0.jpg`).

In [ ]:
!wget -q --show-progress -O cats_dogs.zip https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q -o cats_dogs.zip
!ls

## 2. Скрипт підготовки і розділення набору даних
Скрипт записується у файл `prepare_dataset.py` (його потрібно здати окремо), а потім запускається.

In [ ]:
%%writefile prepare_dataset.py
"""
prepare_dataset.py — підготовка набору Dogs vs Cats для навчання нейромережі в Keras.

1) знаходить усі зображення котів і собак (підтримує обидва формати архіву:
   PetImages/Cat, PetImages/Dog  — дзеркало Microsoft,  або  train/cat.N.jpg, train/dog.N.jpg — Kaggle);
2) відкидає пошкоджені файли (у наборі є порожні та «биті» зображення);
3) перемішує і ділить на навчальний (70%), оціночний (15%) і тестовий (15%) набори;
4) копіює у структуру каталогів, яку розуміє ImageDataGenerator.flow_from_directory:
      dataset/train/cats, dataset/train/dogs, dataset/val/..., dataset/test/...
"""
import os, shutil, random
from PIL import Image

dataset_dir = 'dataset'                         # куди складаємо результат
train_dir = os.path.join(dataset_dir, 'train')  # каталог з даними для навчання
val_dir = os.path.join(dataset_dir, 'val')      # каталог з даними для перевірки
test_dir = os.path.join(dataset_dir, 'test')    # каталог з даними для тестування
val_data_portion = 0.15                         # частка зображень для перевірки
test_data_portion = 0.15                        # частка зображень для тестування
random.seed(42)


def find_images():
    """Повертає словник {'cats': [шляхи], 'dogs': [шляхи]}."""
    if os.path.isdir('PetImages'):              # дзеркало Microsoft (той самий набір Kaggle)
        lst = lambda d: [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith('.jpg')]
        return {'cats': lst('PetImages/Cat'), 'dogs': lst('PetImages/Dog')}
    if os.path.isdir('train'):                  # оригінальний train.zip з Kaggle: cat.0.jpg, dog.0.jpg ...
        files = os.listdir('train')
        return {'cats': [os.path.join('train', f) for f in files if f.startswith('cat.')],
                'dogs': [os.path.join('train', f) for f in files if f.startswith('dog.')]}
    raise FileNotFoundError('Не знайдено ні PetImages/, ні train/ — розпакуйте архів з набором даних')


def is_valid(path):
    """Перевіряє, що файл — справжнє зображення, яке відкривається."""
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()                        # перевірка цілісності файлу
        with Image.open(path) as img:
            img.convert('RGB').load()           # повне декодування
        return True
    except Exception:
        return False


def create_directory(dir_name):
    """Створює каталог з підкаталогами cats і dogs (старий видаляє)."""
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
    os.makedirs(os.path.join(dir_name, 'cats'))
    os.makedirs(os.path.join(dir_name, 'dogs'))


def copy_images(files, dest_dir, class_name):
    for i, src in enumerate(files):
        shutil.copy2(src, os.path.join(dest_dir, class_name, f'{class_name[:-1]}.{i}.jpg'))


images = find_images()
for cls in images:
    total = len(images[cls])
    images[cls] = [p for p in images[cls] if is_valid(p)]
    print(f'{cls}: знайдено {total}, пошкоджених відкинуто {total - len(images[cls])}')

# однакова кількість зображень кожного класу
nb_images = min(len(images['cats']), len(images['dogs']))
start_val_data_idx = int(nb_images * (1 - val_data_portion - test_data_portion))
start_test_data_idx = int(nb_images * (1 - test_data_portion))

for d in (train_dir, val_dir, test_dir):
    create_directory(d)

for cls, files in images.items():
    random.shuffle(files)
    files = files[:nb_images]
    copy_images(files[:start_val_data_idx], train_dir, cls)
    copy_images(files[start_val_data_idx:start_test_data_idx], val_dir, cls)
    copy_images(files[start_test_data_idx:], test_dir, cls)

for d in (train_dir, val_dir, test_dir):
    print(f'{d}: cats = {len(os.listdir(os.path.join(d, "cats")))}, dogs = {len(os.listdir(os.path.join(d, "dogs")))}')

In [ ]:
!python prepare_dataset.py

## 3. Структура каталогів з даними

In [ ]:
for root, dirs, fs in sorted(os.walk('dataset')):
    depth = root.count(os.sep)
    print('    ' * depth + os.path.basename(root) + '/' + (f'   ({len(fs)} файлів)' if fs else ''))

In [ ]:
plt.figure(figsize=(16, 4))
for i, cls in enumerate(['cats'] * 4 + ['dogs'] * 4):
    d = os.path.join('dataset/train', cls)
    plt.subplot(1, 8, i + 1); plt.imshow(load_img(os.path.join(d, sorted(os.listdir(d))[i]))); plt.axis('off'); plt.title(cls)
plt.show()

## 4. Генератори
`ImageDataGenerator(rescale=1/255)` нормалізує пікселі, `flow_from_directory` читає зображення з каталогу,
зменшує до 150×150 і бере мітку з назви підкаталогу (`cats` = 0, `dogs` = 1). Тестовий генератор без перемішування.

In [ ]:
img_width, img_height = 150, 150      # розмір зображень
input_shape = (img_width, img_height, 3)
batch_size = 32

datagen = ImageDataGenerator(rescale=1. / 255)

train_generator = datagen.flow_from_directory('dataset/train', target_size=(img_width, img_height),
                                              batch_size=batch_size, class_mode='binary')
val_generator = datagen.flow_from_directory('dataset/val', target_size=(img_width, img_height),
                                            batch_size=batch_size, class_mode='binary')
test_generator = datagen.flow_from_directory('dataset/test', target_size=(img_width, img_height),
                                             batch_size=batch_size, class_mode='binary', shuffle=False)
print(train_generator.class_indices)

## 5. Згорткова нейронна мережа з лекції
3 блоки Conv2D + MaxPooling (32, 32, 64 фільтри 3×3) → Flatten → Dense 64 → Dropout 0.5 → 1 нейрон sigmoid.
Функція втрат `binary_crossentropy` (2 класи).

In [ ]:
def build_cnn(optimizer):
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(Conv2D(32, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(32, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

build_cnn('adam').summary()

In [ ]:
def plot_history(history, title):
    h = history.history; ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 2, 1); plt.plot(ep, h['accuracy'], 'b', label='Навчальні'); plt.plot(ep, h['val_accuracy'], 'r', label='Перевірочні')
    plt.title('Точність'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2); plt.plot(ep, h['loss'], 'b', label='Навчальні'); plt.plot(ep, h['val_loss'], 'r', label='Перевірочні')
    plt.title('Втрати'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.show()

results, models = [], {}

def train_and_evaluate(name, optimizer, epochs=15):
    model = build_cnn(optimizer)
    # рання зупинка: якщо втрати на перевірці не зменшуються 3 епохи — стоп, повертаємо найкращі ваги
    early = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    start = time.time()
    history = model.fit(train_generator, epochs=epochs, validation_data=val_generator, callbacks=[early], verbose=1)
    t = time.time() - start
    plot_history(history, name)
    test_loss, test_acc = model.evaluate(test_generator, verbose=0)
    h = history.history
    best_ep = int(np.argmin(h['val_loss']))
    results.append({'оптимізатор': name, 'епох': len(h['loss']), 'найкраща епоха': best_ep + 1,
                    'train acc, %': round(h['accuracy'][best_ep] * 100, 2),
                    'val acc, %': round(h['val_accuracy'][best_ep] * 100, 2),
                    'test acc, %': round(test_acc * 100, 2), 'test loss': round(test_loss, 4),
                    'час, хв': round(t / 60, 1)})
    models[name] = model
    print(f'{name}: точність на тестових даних {test_acc*100:.2f}%')

## 6. Навчання: оптимізатор Adam (як у лекції)

In [ ]:
train_and_evaluate('Adam', 'adam')

## 7. Навчання: оптимізатор RMSprop, learning_rate = 0.001 (п. 24)

In [ ]:
train_and_evaluate('RMSprop (lr=0.001)', RMSprop(learning_rate=0.001))

## 8. Порівняння і збереження кращої моделі

In [ ]:
res = pd.DataFrame(results)
res.to_csv('lab7_4_results.csv', index=False)
best_name = res.loc[res['test acc, %'].idxmax(), 'оптимізатор']
best_model = models[best_name]
print('Краща модель:', best_name)
res

In [ ]:
best_model.save('cats_dogs_cnn.keras')
print('Розмір файлу:', round(os.path.getsize('cats_dogs_cnn.keras') / 2**20, 1), 'МБ')
files.download('cats_dogs_cnn.keras')
files.download('lab7_4_results.csv')
files.download('prepare_dataset.py')

## 9. Розпізнавання власних зображень (п. 23)
По 10 різних котів і собак, прості й складні (різний фон, людина з собакою, кіт у кутку, незвичні пози).
**Назвіть файли з префіксом:** `cat_1.jpg` … `cat_10.jpg`, `dog_1.jpg` … `dog_10.jpg` — тоді програма сама порахує вірні відповіді.
Попередня обробка така сама, як у попередніх прикладах: `load_img(target_size=(150, 150))` → масив → `/255` → додатковий вимір.

In [ ]:
def predict_images(paths, title):
    """Розпізнає зображення: вихід sigmoid > 0.5 -> собака, < 0.5 -> кіт."""
    rows = []
    cols = 5; rows_n = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=(cols * 3.2, rows_n * 3.6))
    for i, p in enumerate(paths):
        img = load_img(p, target_size=(img_width, img_height))
        x = img_to_array(img) / 255
        x = np.expand_dims(x, axis=0)
        prob = float(best_model.predict(x, verbose=0)[0][0])
        answer = 'собака' if prob > 0.5 else 'кіт'
        name = os.path.basename(p).lower()
        truth = 'кіт' if name.startswith('cat') else ('собака' if name.startswith('dog') else None)
        ok = None if truth is None else answer == truth
        conf = prob if prob > 0.5 else 1 - prob
        plt.subplot(rows_n, cols, i + 1); plt.imshow(load_img(p)); plt.axis('off')
        plt.title(f'{answer} ({conf*100:.0f}%)\n{os.path.basename(p)[:24]}', fontsize=9,
                  color='black' if ok is None else ('green' if ok else 'red'))
        rows.append({'файл': os.path.basename(p), 'справжній клас': truth or '?', 'відповідь': answer,
                     'вихід мережі': round(prob, 4), 'впевненість, %': round(conf * 100, 1), 'вірно': ok})
    plt.suptitle(title); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows)
    known = df[df['вірно'].notna()]
    if len(known):
        print(f'{title}: вірно {int(known["вірно"].sum())} з {len(known)}')
    return df

def upload_to(folder):
    os.makedirs(folder, exist_ok=True)
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, os.path.join(folder, name))
    return sorted(os.path.join(folder, f) for f in os.listdir(folder))

In [ ]:
own_paths = upload_to('my_images')
own = predict_images(own_paths, 'Власні зображення')
own

## 10. Складні зображення (п. 25)
5–6 фото, де тварина маленька на строкатому фоні (дерево, трава, ялинка, каміння, людина), які модель розпізнає **неправильно**.
Назви теж з префіксом: `cat_h1.jpg`, `dog_h1.jpg`, …

In [ ]:
hard_paths = upload_to('hard_images')
hard = predict_images(hard_paths, 'Складні зображення')
hard

## 11. Обрізані зображення (п. 26)
Обріжте ці самі фото так, щоб тварина була в центрі й без зайвих елементів, і назвіть **так само + `_crop`**:
`cat_h1_crop.jpg`, `dog_h1_crop.jpg`, … Потім порівнюємо відповіді до і після обрізки.

In [ ]:
crop_paths = upload_to('cropped_images')
cropped = predict_images(crop_paths, 'Обрізані зображення')

# порівняння: оригінал -> обрізаний
compare = hard.merge(cropped.assign(файл=cropped['файл'].str.replace('_crop', '', regex=False)),
                     on='файл', suffixes=(' (оригінал)', ' (обрізаний)'))
compare = compare[['файл', 'справжній клас (оригінал)', 'відповідь (оригінал)', 'впевненість, % (оригінал)',
                   'відповідь (обрізаний)', 'впевненість, % (обрізаний)']]
compare.to_csv('lab7_4_crop_compare.csv', index=False)
compare

In [ ]:
for name, df in [('lab7_4_own.csv', own), ('lab7_4_hard.csv', hard)]:
    df.to_csv(name, index=False)
!zip -q -r lab7_4_images.zip my_images hard_images cropped_images
for f in ['lab7_4_own.csv', 'lab7_4_hard.csv', 'lab7_4_crop_compare.csv', 'lab7_4_images.zip']:
    files.download(f)

## Висновки (завдання 4)
*(заповніть після запуску)* Точність на тестовому наборі ...; Adam проти RMSprop ...; перенавчання ...;
власні зображення ... з 20; складні зображення — чому помилки ...; після обрізки ...